In [ ]:
!pip -q install transformers datasets torch scikit-learn tqdm protobuf==3.20.3

In [ ]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, classification_report, accuracy_score
from torch.cuda.amp import autocast, GradScaler 
import logging
import pandas as pd

# --- Setup Logging ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration ---
CONFIG = {
    "model_name": "RoBERTa-Large", 
    "max_length": 512,
    "batch_size": 4,
    "lr": 2e-5,
    "epochs": 15, 
    "warmup_ratio": 0.1,
    "seed": 42,
    # Filtered relations for relevance to emotion attribution
    "EXCLUDED_RELATIONS": {'no_relation', 'unanswerable', 'per:alternate_names', 'per:place_of_birth'},
    "use_fp16": True 
}

# --- Set Seed for Reproducibility ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

# --- Kaggle Paths ---
# Load data from the specified input directory
DATA_PATH = "/kaggle/input/data-dialogre/data_dialogre"
# Save model artifacts to the standard Kaggle output directory
KAGGLE_SAVE_PATH = "/kaggle/working"

In [ ]:
# Cell 2: Data Preprocessing and Filtering (UPDATED to Save Processed Data)

class DialogREProcessor:
    def __init__(self, tokenizer, excluded_relations):
        self.tokenizer = tokenizer
        self.excluded_relations = excluded_relations
        self.special_tokens = {
            's_start': '[SS]', 's_end': '[/SS]', 
            'o_start': '[OS]', 'o_end': '[/OS]'
        }
        
        num_added = self.tokenizer.add_special_tokens({
            'additional_special_tokens': list(self.special_tokens.values())
        })
        if num_added > 0:
            logger.info(f"Added {num_added} special tokens to tokenizer.")

    def process_data(self, file_path):
        """Loads and processes the DialogRE data, filtering out unwanted relations."""
        if not os.path.exists(file_path):
            logger.error(f"File not found: {file_path}")
            # Try a slightly different path construction if the initial one failed
            if os.path.exists(file_path.replace('/data_dialogre', '/data_dialogre/data_v2/en')):
                 file_path = file_path.replace('/data_dialogre', '/data_dialogre/data_v2/en')
                 logger.info(f"Trying alternative path: {file_path}")
            else:
                 return []
            
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        samples = []
        
        for conv in data:
            dialogue = conv[0]
            relations = conv[1]
            
            dialogue_text_list = []
            for item in dialogue:
                dialogue_text_list.append(item)
            
            for rel in relations:
                x_text = rel['x']
                y_text = rel['y']
                r_types = rel['r']
                
                for r_label in r_types:
                    if r_label in self.excluded_relations:
                        continue 
                        
                    processed_text = self._insert_markers(dialogue_text_list, x_text, y_text)
                    
                    samples.append({
                        'text': processed_text,
                        'subject': x_text,
                        'object': y_text,
                        'relation': r_label
                    })
        
        logger.info(f"Processed {len(samples)} samples from {os.path.basename(file_path)}.")
        return samples

    def _insert_markers(self, dialogue_list, subj, obj):
        """Inserts entity markers ([SS], [OS]) into the dialogue text."""
        full_text = " ".join(dialogue_list)
        
        s_mark = f"{self.special_tokens['s_start']} {subj} {self.special_tokens['s_end']}"
        o_mark = f"{self.special_tokens['o_start']} {obj} {self.special_tokens['o_end']}"
        
        processed = full_text.replace(subj, s_mark, 1).replace(obj, o_mark, 1)
        return processed

# --- Data Loading and Tokenizer Initialization ---
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
processor = DialogREProcessor(tokenizer, CONFIG['EXCLUDED_RELATIONS'])

# Paths to the JSON files (Kaggle directory structure assumed)
train_file = os.path.join(DATA_PATH, "train.json")
dev_file = os.path.join(DATA_PATH, "dev.json")
test_file = os.path.join(DATA_PATH, "test.json")

# Process Data
logger.info("Starting data processing and filtering...")
train_data = processor.process_data(train_file)
dev_data = processor.process_data(dev_file)
test_data = processor.process_data(test_file)

# -------------------------------------------------------------
# NEW: Saving Processed Data Artifacts to Disk
# -------------------------------------------------------------
def save_processed_data(data, filename):
    filepath = os.path.join(KAGGLE_SAVE_PATH, filename)
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=4)
    logger.info(f"Processed data saved to {filepath}")

save_processed_data(train_data, 'processed_train_data.json')
save_processed_data(dev_data, 'processed_dev_data.json')
save_processed_data(test_data, 'processed_test_data.json')

# CRITICAL FIX: Create Label Mapping from the union of all processed data
all_relations_in_data = set()
for s in train_data + dev_data + test_data:
    all_relations_in_data.add(s['relation'])
    
all_labels = sorted(list(all_relations_in_data))

label2id = {l: i for i, l in enumerate(all_labels)}
id2label = {i: l for l, i in label2id.items()}

# Save the label mapping as a JSON file as well
label_map_path = os.path.join(KAGGLE_SAVE_PATH, 'label_map.json')
with open(label_map_path, 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=4)
logger.info(f"Label map saved to {label_map_path}")

logger.info(f"Total Labels (Post-Filtering): {len(label2id)}")
logger.info(f"Labels to be trained on: {all_labels}")

In [ ]:
class REDataset(Dataset):
    """
    Standard PyTorch Dataset for Relation Extraction.
    """
    def __init__(self, samples, tokenizer, label2id, max_len=512):
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len
        self.samples = samples
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        text = item['text']
        
        label = self.label2id.get(item['relation']) 
        
        if label is None:
            raise ValueError(
                f"FATAL ERROR: Relation '{item['relation']}' found in data at index {idx} "
                f"is not present in the training label map."
            )
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class UniversalREModel(nn.Module):
    """
    Transformer model designed for Relation Extraction using entity marker embeddings.
    """
    def __init__(self, model_name, num_labels, tokenizer):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.bert.resize_token_embeddings(len(tokenizer)) 
        
        self.dropout = nn.Dropout(0.1)
        
        self.s_start_id = tokenizer.convert_tokens_to_ids('[SS]')
        self.o_start_id = tokenizer.convert_tokens_to_ids('[OS]')
        
        self.classifier = nn.Linear(self.config.hidden_size * 2, num_labels)
        
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        batch_size, seq_len, hidden_size = sequence_output.shape
        
        s_emb = torch.zeros(batch_size, hidden_size).to(input_ids.device)
        o_emb = torch.zeros(batch_size, hidden_size).to(input_ids.device)
        
        for i in range(batch_size):
            s_idx = (input_ids[i] == self.s_start_id).nonzero(as_tuple=True)[0]
            o_idx = (input_ids[i] == self.o_start_id).nonzero(as_tuple=True)[0]
            
            if s_idx.numel() > 0:
                s_emb[i] = sequence_output[i, s_idx[0], :]
            else:
                s_emb[i] = sequence_output[i, 0, :]
                
            if o_idx.numel() > 0:
                o_emb[i] = sequence_output[i, o_idx[0], :]
            else:
                o_emb[i] = sequence_output[i, 0, :]

        combined = torch.cat([s_emb, o_emb], dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        
        loss = None
        if labels is not None:
            loss = self.compute_loss(logits, labels)
            
        return {"loss": loss, "logits": logits}

    def compute_loss(self, logits, labels, gamma=2.0, alpha=0.25):
        """Focal Loss implementation."""
        # Note: If you want to disable Focal Loss, manually change this or update CONFIG
        if False: # Placeholder for CONFIG['use_focal_loss'] if it existed
            return nn.CrossEntropyLoss()(logits, labels)
            
        ce_loss = nn.CrossEntropyLoss(reduction='none')(logits, labels)
        pt = torch.exp(-ce_loss)
        focal_loss = (alpha * (1 - pt)**gamma * ce_loss).mean() 
        return focal_loss

In [ ]:
# --- EarlyStopper Class for Regularization ---

class EarlyStopper:
    """
    Stops training when the monitored metric (Macro F1) hasn't improved
    for a specified number of epochs (patience).
    """
    def __init__(self, patience=3, min_delta=0.001): # Increased patience based on performance feedback
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False

    def __call__(self, val_score):
        if val_score > self.best_score + self.min_delta:
            self.best_score = val_score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                
        return self.early_stop

# -------------------------------------------------------------------

def main():
    # 1. Initialize Datasets and Loaders
    train_dataset = REDataset(train_data, tokenizer, label2id, CONFIG['max_length'])
    val_dataset = REDataset(dev_data, tokenizer, label2id, CONFIG['max_length'])
    test_dataset = REDataset(test_data, tokenizer, label2id, CONFIG['max_length'])

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], num_workers=2)
    
    logger.info(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

    # 2. Initialize Model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = UniversalREModel(CONFIG['model_name'], len(label2id), tokenizer)
    model.to(device)

    # 3. Optimizer and Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    total_steps = len(train_loader) * CONFIG['epochs']
    num_warmup_steps = int(CONFIG['warmup_ratio'] * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=num_warmup_steps, 
        num_training_steps=total_steps
    )
    
    # 4. Initialize Early Stopper and GradScaler
    early_stopper = EarlyStopper(patience=5, min_delta=0.001) 
    # Initialize GradScaler only if FP16 is enabled in CONFIG
    scaler = GradScaler() if CONFIG.get('use_fp16', False) else None 
    
    # 5. Training Loop
    best_f1 = -1.0
    MODEL_SAVE_PATH = os.path.join(KAGGLE_SAVE_PATH, 'RoBERTa-Large_model.pth')
    
    print(f"Model will be saved permanently to: {MODEL_SAVE_PATH}")
    
    for epoch in range(CONFIG['epochs']):
        # --- TRAINING STEP ---
        model.train()
        train_loss = 0
        
        for i, batch in enumerate(train_loader):
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # --- Mixed Precision (FP16) or Standard (FP32) Forward Pass ---
            with autocast(enabled=CONFIG['use_fp16']):
                outputs = model(input_ids, mask, labels=labels)
                loss = outputs['loss']
            
            if CONFIG['use_fp16']:
                # Backpropagate using the scaler (FP16/AMP)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
                scaler.step(optimizer)
                scaler.update() 
            else:
                # Standard backpropagation (FP32)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
            scheduler.step()
            train_loss += loss.item()
            
            if (i + 1) % 300 == 0:
                print(f"Epoch {epoch+1} | Step {i+1}/{len(train_loader)} | Loss: {train_loss/(i+1):.4f}")
            
        # 6. Validation (Validation is always done in FP32/no_grad)
        model.eval()
        preds, truths = [], []
        
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            with torch.no_grad():
                outputs = model(input_ids, mask)
            
            logits = outputs['logits']
            pred = torch.argmax(logits, dim=-1)
            
            preds.extend(pred.cpu().numpy())
            truths.extend(labels.cpu().numpy())
        
        macro_f1 = f1_score(truths, preds, average='macro', zero_division=0)
        val_accuracy = accuracy_score(truths, preds) 
        
        print(f"\n--- Epoch {epoch+1} Summary ---")
        print(f"Avg Train Loss: {train_loss/len(train_loader):.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        print(f"Validation Macro F1: {macro_f1:.4f}")
        
        # 7. Early Stopping and Checkpoint
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), MODEL_SAVE_PATH) 
            print(f"Saved New Best Model to {MODEL_SAVE_PATH}")
            
        if early_stopper(macro_f1):
            print(f"\n*** Early stopping triggered after {epoch+1} epochs! ***")
            break # Exit the training loop
            
    # 8. Final Test Evaluation and Saving Artifacts
    logger.info("Starting final test evaluation...")
    
    # Load the best model weights saved to disk
    if os.path.exists(MODEL_SAVE_PATH):
        model.load_state_dict(torch.load(MODEL_SAVE_PATH)) 
        logger.info(f"Loaded best model with F1: {best_f1:.4f}")
    else:
        logger.error("Best model weights not found. Using the last epoch's weights.")
        
    model.eval()
    preds, truths = [], []
    
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device) 
        
        with torch.no_grad():
            outputs = model(input_ids, mask)
        
        preds.extend(torch.argmax(outputs['logits'], dim=-1).cpu().numpy())
        truths.extend(labels.cpu().numpy())
        
    # --- CALCULATE AND PRINT FULL MATRIX EVALUATION ---
    test_accuracy = accuracy_score(truths, preds)
    test_macro_f1 = f1_score(truths, preds, average='macro', zero_division=0)
    report = classification_report(truths, preds, target_names=list(label2id.keys()), zero_division=0) 
    
    print("\n==============================================")
    print("           FINAL TEST CLASSIFICATION REPORT")
    print("==============================================")
    print(f"Test Accuracy: {test_accuracy:.4f} (Overall Correct Predictions)")
    print(f"Test Macro F1: {test_macro_f1:.4f} (Average F1 across all classes)")
    print(report)

    # --- SAVE ARTIFACTS TO KAGGLE WORKING DIRECTORY ---
    
    # 1. Save Evaluation Report
    report_file_path = os.path.join(KAGGLE_SAVE_PATH, 'test_evaluation_report.txt')
    
    with open(report_file_path, 'w') as f:
        f.write("RELATION EXTRACTION TEST EVALUATION REPORT\n")
        f.write("================================================\n")
        f.write(f"Model: {CONFIG['model_name']} (Filtered for Emotion Relevance)\n")
        f.write(f"Best Validation Macro F1: {best_f1:.4f} (Used for Test Evaluation)\n")
        f.write(f"Test Accuracy: {test_accuracy:.4f}\n")
        f.write(f"Test Macro F1: {test_macro_f1:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(report)
        
    print(f"Evaluation report saved permanently to: {report_file_path}")

if __name__ == "__main__":
    main()